## Load the required packages

In [1]:
from CCSFormer.Prediction import *
import wget
import time
import sys, os

## Download the CCSFormer models

In [2]:
url_base = "https://github.com/yuxuanliao/row_swap/releases/download/v1.0.0/CCSFormer_base.pth"
CCSFormer_base_PATH = wget.download(url_base, ".")
url_precise = "https://github.com/yuxuanliao/row_swap/releases/download/v1.0.0/CCSFormer_precise.pth"
CCSFormer_precise_PATH = wget.download(url_precise, ".")

## Config

In [3]:
VOCAB_PATH = "data/vocab.pkl"
AD_CACHE_PATH = "data/training_fingerprint.pkl"

with open(VOCAB_PATH, "rb") as f:
    dictionary = dill.load(f)

with open(AD_CACHE_PATH, "rb") as f:
    ad_index = dill.load(f)

## Load models (CCSFormer_base.pth and CCSFormer_precise.pth)

In [4]:
base_model_f1 = TransformerModel(
   vocab_size=14073, embed_dim=128, num_heads=4, en_layers=2, ff_dim=512, max_len=250
).to(DEVICE)

ccs_model_f1 = CCSPredictor(base_model_f1).to(DEVICE)
ccs_model_f1 = load_model(ccs_model_f1, CCSFormer_base_PATH)

base_model_f2 = TransformerModel(
    vocab_size=14073, embed_dim=128, num_heads=4, en_layers=2, ff_dim=512, max_len=250
).to(DEVICE)

ccs_model_f2_base = CCSPredictor(base_model_f2).to(DEVICE)
ccs_model_f2 = CCSPredictor3(ccs_model_f2_base).to(DEVICE)
ccs_model_f2 = load_model(ccs_model_f2, CCSFormer_precise_PATH)

## Load the file that needs to predict CCS values

In [5]:
ifilepath = './data/ExternalTestSet1.csv'

## Path to save the predicted CCS values

In [6]:
ofilepath = './data/ExternalTestSet1-Prediction.csv'

## Use CCSFormer for CCS prediction

In [7]:
sys.stderr = open(os.devnull, "w")
start = time.time()

results = predict_external(
    ifilepath,
    ccs_model_f1,
    ccs_model_f2,
    dictionary,
    ad_index,
    ofilepath
)

elapsed = time.time() - start
print(f"⏱️  Running Time: {elapsed:.2f} seconds")

## CCS predicted completed
R2 Score : 0.9626313568531171
Mean Absolute Error : 4.600829957017108
Median Absolute Error : 2.9888688198725077
Median Relative Error : 1.6260532409025716 %
Mean Relative Error : 2.321369058043848 %
Root Mean Squared Error : 7.328520173863576
Mean Squared Error : 53.70720793872542
⏱️  Running Time: 380.83 seconds
